## Classical Models

### Download 4 Real Timeseries from Yahoo
Download daily historical data for Apple Inc. (AAPL) stock, the S&P 500 Index, the USD/JPY FX spot rate, and the Gold commodity spot price from 2001-01-01 to 2021-01-01

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from empirical_tests import run_tests, plot_daily_returns, acf_plots_block, qq_plot
import torch
import torch.nn as nn
import torch.nn.functional as F
from metrics import calculate_scores, real_data_loading
from timeseries import log_returns, download_data

start_date = "2001-01-01"
end_date = "2021-01-01"

 # Single stock example: Apple Inc. (AAPL)
single_stock = download_data("AAPL", start_date=start_date, end_date=end_date)

# Stock index example: S&P 500 Index (^GSPC)
stock_index = download_data("^GSPC", start_date=start_date, end_date=end_date)

# FX spot example: USD/JPY (JPY=X)
fx_spot = download_data("JPY=X", start_date=start_date, end_date=end_date)

# Commodity spot example: Gold (GC=F)
commodity_spot = download_data("GC=F", start_date=start_date, end_date=end_date)

In [ ]:
ts_lst = [single_stock['Close'].to_numpy().squeeze(), stock_index['Close'].to_numpy().squeeze(), fx_spot['Close'].to_numpy().squeeze(), commodity_spot['Close'].to_numpy().squeeze()]

In [ ]:
single_stock_returns = log_returns(single_stock['Close']).squeeze()
stock_index_returns = log_returns(stock_index['Close']).squeeze()
fx_spot_returns = log_returns(fx_spot['Close']).squeeze()
commodity_spot_returns = log_returns(commodity_spot['Close']).squeeze()

In [ ]:
lst_returns = [single_stock_returns, stock_index_returns, fx_spot_returns, commodity_spot_returns]
lst_labels = ['Apple', 'S&P 500', 'USDJPY', 'Gold']

In [ ]:
ssr2 = real_data_loading(single_stock_returns, 32)
sir2 = real_data_loading(stock_index_returns, 32)
fxr2 = real_data_loading(fx_spot_returns, 32)
csr2 = real_data_loading(commodity_spot_returns, 32)

In [ ]:
real_lst = [ssr2, sir2, fxr2, csr2]

### Gaussian Random Walk

Fitting the model is trivial

In [ ]:
def fit_random_walk(returns):    
    rw_mean = returns.mean()
    rw_std = returns.std()
    return rw_mean, rw_std

In [ ]:
std_devs = [fit_random_walk(x)[1] for x in lst_returns]

Generate sequences the same length as the original time series

In [ ]:
series_len = len(single_stock_returns)

In [ ]:
grw_returns = [np.random.normal(0, istd, series_len) for istd in std_devs]

In [ ]:
grw_filename = 'GRWReturns.png'

#### Daily Returns

In [ ]:
plot_daily_returns(grw_returns, stock_index.index[:-1], grw_filename, lst_labels)

#### Run Empirical Tests

In [ ]:
df = run_tests(grw_returns, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
grwacf_filename = 'GRWACFs.png'
acf_plots_block(grw_returns, lst_labels, grwacf_filename)

#### QQ Plot

In [ ]:
grw_qq_filename = 'GRWQQ.png'
qq_plot(lst_returns, grw_returns, grw_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
syn_lst = list()
for isym in grw_returns:
    syn_lst.append(real_data_loading(isym, 32))

In [ ]:
import torch
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
grw_df = calculate_scores(real_lst, syn_lst, lst_labels, device)

In [ ]:
grw_df

In [ ]:
grw_df.to_latex()

### Heston

To fit the Heston model the svolfit package is used

In [ ]:
import svolfit

In [ ]:
dt = 1/252

In [ ]:
heston_params = [svolfit.svolfit(x, dt, model='Heston', method='grid') for x in ts_lst]

In [ ]:
heston_dict_lst = [x[2] for x in heston_params]

In [ ]:
heston_param_lst = [(x['rep_alpha'], x['rep_theta'], x['rep_eta'], x['rep_rho'], x['rep_v0']) for x in heston_dict_lst]

In [ ]:
def simulate_heston(heston_params, num_steps, num_paths, dt, r=0, s0=1):
    kappa, theta, sigma, rho, v0 = heston_params
    w1 = np.random.randn(num_paths, num_steps)
    w2 = rho * w1 + np.sqrt(1 - rho ** 2) * np.random.randn(num_paths, num_steps)

    s = np.zeros((num_paths, num_steps + 1))
    v = np.zeros((num_paths, num_steps + 1))

    s[:, 0] = s0
    v[:, 0] = v0

    for t in range(1, num_steps + 1):
        dv = kappa * (theta - np.maximum(v[:, t - 1], 0)) * dt + sigma * \
            np.sqrt(np.maximum(v[:, t - 1], 0)) * np.sqrt(dt) * w2[:, t - 1]
        ds = r * s[:, t - 1] * dt + np.sqrt(np.maximum(v[:, t - 1], 0)) * s[:, t - 1] * \
            np.sqrt(dt) * w1[:, t - 1]

        v[:, t] = v[:, t - 1] + dv
        s[:, t] = s[:, t - 1] * np.exp(ds)

    return s, v

In [ ]:
heston_paths = [simulate_heston(x, series_len, 1, dt) for x in heston_param_lst]

In [ ]:
heston_paths[0][0].squeeze()

In [ ]:
heston_returns = [log_returns(x[0].squeeze()) for x in heston_paths]

In [ ]:
heston_returns

#### Daily Returns

In [ ]:
heston_filename = 'HestonReturns.png'
plot_daily_returns(heston_returns, stock_index.index[:-1], heston_filename, lst_labels)

#### Run Empirical Tests

In [ ]:
df = run_tests(heston_returns, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
heston_filename = 'HestonACFs.png'
acf_plots_block(heston_returns, lst_labels, heston_filename)

#### QQ Plot

In [ ]:
heston_qq_filename = 'HestonQQ.png'
qq_plot(lst_returns, heston_returns, heston_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
syn_lst = list()
for isym in heston_returns:
    syn_lst.append(real_data_loading(isym, 32))

In [ ]:
heston_df = calculate_scores(real_lst, syn_lst, lst_labels, device)

In [ ]:
heston_df

In [ ]:
heston_df.to_latex()

### ARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import statsmodels as sm

In [ ]:
def grid_search_arima(returns, pmax, qmax):
    best_aic = np.inf
    best_p = None
    best_q = None
    best_model = None

    for p in range(1, pmax + 1):
        for q in range(1, qmax + 1):
            try:
                model = ARIMA(returns, order=(p, 0, q)).fit()
                if model.aic < best_aic:
                    best_aic = model.aic
                    best_p = p
                    best_q = q
                    best_model = model
            except:
                continue

    return best_model, best_p, best_q

In [ ]:
pmax = 10
qmax = 10

In [ ]:
arima_model_lst = [grid_search_arima(x, pmax, qmax) for x in lst_returns]

In [ ]:
arima_fit = [x[0] for x in arima_model_lst]

In [ ]:
arima_fit[0].summary()

In [ ]:
residuals = pd.DataFrame(arima_fit[0].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
arima_fit[1].summary()

In [ ]:
residuals = pd.DataFrame(arima_fit[1].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
arima_fit[2].summary()

In [ ]:
residuals = pd.DataFrame(arima_fit[2].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
arima_fit[3].summary()

In [ ]:
residuals = pd.DataFrame(arima_fit[3].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
arima_sim = [x.simulate(series_len, anchor='end') for x in arima_fit]

#### Daily Returns

In [ ]:
arima_filename = 'ARIMAReturns.png'
plot_daily_returns(arima_sim, stock_index.index[:-1], arima_filename, lst_labels)

#### Run Empirical Tests

In [ ]:
df = run_tests(arima_sim, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
arima_filename = 'ARIMAACFs.png'
acf_plots_block(arima_sim, lst_labels, arima_filename)

#### QQ Plot

In [ ]:
arima_qq_filename = 'ARIMAQQ.png'
qq_plot(lst_returns, arima_sim, arima_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
syn_lst = list()
for isym in arima_sim:
    syn_lst.append(real_data_loading(isym, 32))

In [ ]:
arima_df = calculate_scores(real_lst, syn_lst, lst_labels, device)

In [ ]:
arima_df

In [ ]:
arima_df.to_latex()

### GARCH

In [ ]:
from arch import arch_model

In [ ]:
pmax = 10
qmax = 10

In [ ]:
def grid_search_garch(returns, pmax, qmax):
    best_aic = np.inf
    best_p = None
    best_q = None
    best_model = None

    for p in range(1, pmax + 1):
        for q in range(1, qmax + 1):
            try:
                model = arch_model(returns, vol='Garch', p=p, o=0, q=q).fit()
                if model.aic < best_aic:
                    best_aic = model.aic
                    best_p = p
                    best_q = q
                    best_model = model
            except:
                continue

    return best_model, best_p, best_q

In [ ]:
garch_model_lst = [grid_search_garch(x, pmax, qmax) for x in lst_returns]

In [ ]:
garch_fit = [x[0] for x in garch_model_lst]
garch_p = [x[1] for x in garch_model_lst]
garch_q = [x[2] for x in garch_model_lst]

In [ ]:
garch_fit[0].summary()

In [ ]:
residuals = pd.DataFrame(garch_fit[0].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
garch_fit[1].summary()

In [ ]:
residuals = pd.DataFrame(garch_fit[1].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
garch_fit[2].summary()

In [ ]:
residuals = pd.DataFrame(garch_fit[2].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
garch_fit[3].summary()

In [ ]:
residuals = pd.DataFrame(garch_fit[3].resid)
residuals.plot()

In [ ]:
residuals.plot(kind='kde')

In [ ]:
sim_mod = [arch_model(None, p=x, o=0, q=y) for x, y in zip(garch_p, garch_q)]

In [ ]:
Southgate Stationgarch_sim = [y.simulate(x.params, series_len) for x, y in zip(garch_fit, sim_mod)]

#### Daily Returns

In [ ]:
garch_filename = 'GARCHReturns.png'
plot_daily_returns(garch_sim, stock_index.index[:-1], garch_filename, lst_labels)

In [ ]:
garch_sim_data = [x['data'] for x in garch_sim]

#### Run Empirical Tests

In [ ]:
df = run_tests(garch_sim_data, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
garch_filename = 'GARCHACFs.png'
acf_plots_block(garch_sim_data, lst_labels, garch_filename)

#### QQ Plot

In [ ]:
garch_qq_filename = 'GARCHQQ.png'
qq_plot(lst_returns, garch_sim, garch_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
syn_lst = list()
for isym in garch_sim:
    syn_lst.append(real_data_loading(isym.to_numpy()[:,0], 32))

In [ ]:
garch_df = calculate_scores(real_lst, syn_lst, lst_labels, device)

In [ ]:
garch_df

In [ ]:
garch_df.to_latex()

In [ ]:
garch_sim[0].to_numpy()[:,0]